# XENONnT SR2 LowER — Run Tagging & Quality Analysis

**Integrated pipeline demonstration**  
This notebook walks through the complete SR2 LowER analysis workflow:

1. **Event-rate extraction** — process raw XENONnT data per run, compute physics event rates
2. **Detector quality scoring** — unsupervised anomaly detection via Isolation Forest  
3. **Publication-grade visualization** — evolution plots, anomaly diagnostics, trend monitoring

---

**Environment**: XENONnT CVMFS / singularity container  
**Data period**: Oct 2023 – Apr 2025 (SR2)  
**Cluster**: University of Chicago Midway3  

## 0. Imports & Setup

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# Add the project directory to the path so we can import the core scripts
PROJECT_DIR = '/scratch/midway3/jiafu/SR2_LowER/run_tagging_lower'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# XENONnT software stack
import strax
import straxen

# Try loading cutax for offline processing
try:
    import cutax
    print('✓ cutax available — using xenonnt_offline context')
    HAS_CUTAX = True
except ImportError:
    print('⚠ cutax not available — using xenonnt_online context')
    HAS_CUTAX = False

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

print('\n✓ All imports ready.')

---
## 1. Event-Rate Extraction

Compute per-run physics event rates using `compute_event_rates.py`.
For each run we extract five event categories and normalize by livetime.

In [ ]:
# Import the rate-computation module
from compute_event_rates import load_run_metadata, compute_rates

# Paths
RUN_INFO_CSV = (
    '/scratch/midway3/jiafu/SR2_LowER/SRs_Analysis_Hub/SR2/'
    'data_organization/run_tagging/results/sr2_run_tagging_info_0.0.5.csv'
)

# --- Demo: look up metadata for a specific run ---
DEMO_RUN = '054585'
livetime, start_date = load_run_metadata(RUN_INFO_CSV, DEMO_RUN)

print(f'Run {DEMO_RUN}:')
print(f'  Start date : {start_date}')
print(f'  Livetime   : {livetime:.1f} s  ({livetime/3600:.2f} h)')

### 1.1 Set up the XENONnT strax context

The context gives access to all processed data types. We add the standard data directories.

In [ ]:
if HAS_CUTAX:
    st = cutax.contexts.xenonnt_offline()
else:
    st = straxen.contexts.xenonnt_online()

# Add standard data storage paths
st.storage += [
    strax.DataDirectory(p, readonly=True)
    for p in [
        '/project2/lgrandi/xenonnt/processed/',
        '/project/lgrandi/xenonnt/processed/',
    ]
]

print(f'Strax context ready.')
print(f'Storage locations: {len(st.storage)}')

### 1.2 Load event data and compute rates

We load `event_info` (and `event_shadow` if available), apply physics masks, and compute livetime-normalized rates.

In [ ]:
run_id = DEMO_RUN

# Load targets
targets = ['event_info']
if st.is_stored(run_id, 'event_shadow'):
    targets.append('event_shadow')
    print('  Including event_shadow')

df = st.get_df(run_id, targets=tuple(targets))
print(f'  Loaded {len(df):,} events from run {run_id}')

# Derived quantities
df['r2'] = df['x']**2 + df['y']**2
s1_raw = df['s1_area'].fillna(0)
s2_raw = df['s2_area'].fillna(0)

# Physics category masks
categories = {
    'Gate_Event':    (df['drift_time'] > 0) & (df['drift_time'] < 8e3),
    'Cathode_Event': (
        ((df['drift_time'] > 1.8e6) & (df['drift_time'] < 2.5e6))
        | ((df['z'] > -150) & (df['z'] < -145))
        | ((s1_raw > 1000) & (s2_raw < 200))
    ),
    'S1_Only_Heavy': (s1_raw < 100) & (s2_raw < 100),
    'S2_Only_SE':    (s1_raw < 10) & (s2_raw < 200),
    'Wall_Event':    (df['r2'] > 3800),
}

# Compute and display rates
print(f'\n{"Category":<18} {"Count":>8}  {"Rate (Hz)":>10}')
print('-' * 40)
rates = {}
for name, mask in categories.items():
    count = int(mask.sum())
    rate = count / livetime
    rates[name] = rate
    print(f'{name:<18} {count:>8,}  {rate:>10.4f}')

print(f'\n✓ Rate computation complete for run {run_id}')

### 1.3 Quick look at the master rate table

The batch processing (via `submit_batch.sh`) accumulates all runs into a single CSV.

In [ ]:
rates_csv = os.path.join(PROJECT_DIR, 'sr2_master_run_rates.csv')

if os.path.exists(rates_csv):
    df_rates = pd.read_csv(rates_csv)
    df_rates['Start_Date'] = pd.to_datetime(df_rates['Start_Date'])
    print(f'Master rate table: {len(df_rates)} runs')
    print(f'Date range: {df_rates["Start_Date"].min().date()} → {df_rates["Start_Date"].max().date()}')
    display(df_rates.head())
else:
    print(f'Rate CSV not yet generated. Run submit_batch.sh first.')
    print(f'Expected path: {rates_csv}')

---
## 2. Detector Quality Scoring

Using `compute_quality_scores.py`, we merge detector parameters with event rates and apply **Isolation Forest** (unsupervised anomaly detection) to assign each run a 0–100 quality score.

Three scoring modes are available:
- **Single** — one IF model on per-run features
- **Batch** — rolling-window mean features → IF
- **Consensus** — multi-window voting with MAD-based threshold (most robust)

In [ ]:
from compute_quality_scores import QualityAnalyzer

# Initialize the analyzer
analyzer = QualityAnalyzer(
    run_info_path=RUN_INFO_CSV,
    rates_path=os.path.join(PROJECT_DIR, 'sr2_master_run_rates_with_mode.csv'),
    deadtime_path=None,  # v2+ mode: no deadtime dependency
)

# Load and merge all data sources
df_merged = analyzer.load_and_merge()
print(f'\nMerged dataset: {len(df_merged)} runs with both metadata and rates')

### 2.1 Feature Selection

The analyzer automatically identifies numerical physical features while excluding metadata columns (IDs, timestamps, binning info, etc.).

In [ ]:
features = analyzer.extract_features(analyzer.df)
print(f'Features selected for ML: {len(features)}')
print('\nFirst 20 features:')
for f in features[:20]:
    print(f'  • {f}')

### 2.2 Single-Run Quality Scoring

Train one Isolation Forest (150 estimators) on all per-run features. The decision function is rescaled to 0–100.

In [ ]:
df_single = analyzer.score_single()

print(f'Quality scores computed for {len(df_single)} runs')
print(f'\nScore distribution:')
print(f'  Mean   : {df_single["quality_score"].mean():.1f}')
print(f'  Median : {df_single["quality_score"].median():.1f}')
print(f'  Std    : {df_single["quality_score"].std():.1f}')
print(f'  Min    : {df_single["quality_score"].min():.1f}')
print(f'  Max    : {df_single["quality_score"].max():.1f}')

In [ ]:
# Quick histogram of quality scores
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_single['quality_score'], bins=50, color='steelblue', edgecolor='black', alpha=0.8)
ax.axvline(20, color='red', linestyle='--', linewidth=2, label='Anomaly threshold (20)')
ax.set_xlabel('Quality Score (0–100)', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Runs', fontsize=14, fontweight='bold')
ax.set_title('Single-Run Quality Score Distribution', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

### 2.3 Consensus Scoring (most robust)

Train multiple IF models at different rolling-window sizes. Each window votes on whether a run is anomalous (MAD-based threshold). A run is flagged bad only if a majority of windows agree.

This significantly reduces false positives compared to single-model scoring.

In [ ]:
# Re-create analyzer for a clean start
analyzer2 = QualityAnalyzer(
    run_info_path=RUN_INFO_CSV,
    rates_path=os.path.join(PROJECT_DIR, 'sr2_master_run_rates_with_mode.csv'),
)
analyzer2.load_and_merge()

# Consensus with 5 window sizes
WINDOW_SIZES = [1, 2, 4, 8, 10]
df_consensus = analyzer2.score_consensus(
    window_sizes=WINDOW_SIZES,
    k_mad=4.5,       # MAD multiplier (~3σ equivalent)
    vote_ratio=0.5,  # >50% of windows must flag a run
)

n_bad = df_consensus['is_consensus_bad'].sum()
print(f'\nConsensus result: {n_bad} / {len(df_consensus)} runs flagged as anomalous')

In [ ]:
# Show the most anomalous runs
bad_runs = df_consensus[df_consensus['is_consensus_bad']][
    ['number', 'quality_score', 'anomaly_votes', 'is_consensus_bad']
].sort_values('quality_score')

print('Top 10 most anomalous runs (by consensus):')
display(bad_runs.head(10))

---
## 3. Publication-Grade Visualization

All plotting is handled by `plot_evolution.py`, which provides five subcommands via CLI and exposes the underlying functions for inline use.

We demonstrate the key plot types below.

In [ ]:
# Import all plot functions from the visualization module
from plot_evolution import (
    set_publication_style,
    plot_rate_evolution,
    plot_quality_trend,
    plot_all_features,
    plot_anomaly_diagnostics,
    plot_run_diagnostic,
    merge_intervals,
    _build_mode_spans,
    _paint_backgrounds,
    _read_quality_file,
    _detect_mode_column,
    CALIB_STYLES,
    BACKGROUND_ALPHA,
)

set_publication_style()
print('✓ All plot functions imported.')

### 3.1 Event-Rate Evolution

5-panel stacked timeseries showing how each event category evolves over the SR2 data-taking period. Calibration campaigns are shaded as colored backgrounds.

In [ ]:
# Quick inline version of rate evolution using the loaded rates DataFrame
if 'df_rates' in dir() and len(df_rates) > 0:
    RATE_COLS = {
        'Gate_Event_Rate_Hz': 'Gate Events',
        'Cathode_Event_Rate_Hz': 'Cathode Events',
        'S1_Only_Heavy_Rate_Hz': 'S1-only (Heavy)',
        'S2_Only_SE_Rate_Hz': 'S2-only (SE)',
        'Wall_Event_Rate_Hz': 'Wall Events',
    }
    
    fig, axes = plt.subplots(5, 1, figsize=(14, 18), sharex=True)
    
    for i, (col, label) in enumerate(RATE_COLS.items()):
        ax = axes[i]
        if col in df_rates.columns:
            x, y = df_rates['Start_Date'], df_rates[col]
            ax.plot(x, y, color='#cccccc', linewidth=1.2, zorder=1)
            sc = ax.scatter(x, y, c=y, cmap='RdYlGn', s=30,
                            edgecolors='black', linewidths=0.4, zorder=2)
            cbar = fig.colorbar(sc, ax=ax, pad=0.01)
            cbar.set_label('Hz', fontweight='bold')
            cbar.outline.set_linewidth(1.5)
        ax.set_ylabel('Rate [Hz]', fontweight='bold')
        ax.set_title(label, loc='center', fontsize=14, fontweight='bold', pad=8)
        ax.set_facecolor('#ffffff')
        ax.tick_params(axis='both', which='major', width=1.5, length=5)
    
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.setp(ax.get_xticklabels(), fontweight='bold')
    plt.xlabel('Time', fontsize=14, fontweight='bold')
    plt.suptitle('Evolution of XENONnT SR2 LowER Event Rates',
                 y=0.995, fontsize=20, fontweight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
else:
    print('Rate data not available. Run compute_event_rates.py first.')

### 3.2 Quality-Score Trend

Dual-panel view of detector quality over time and by run number.  
Coloured backgrounds distinguish calibration campaigns (Kr-83m, AmBe, Rn-220, Ar-37, Th-232, neutrons) from science runs.  
Red ✗ markers highlight consensus-flagged anomalous runs.

In [ ]:
# Build a full quality trend plot inline
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches

df_q = df_consensus.copy()
df_info = pd.read_csv(RUN_INFO_CSV)
df_info['number'] = df_info['number'].astype(str).str.zfill(6)
if 'start' in df_info.columns:
    df_info['start'] = pd.to_datetime(df_info['start'], errors='coerce')
if 'start' not in df_q.columns:
    df_q = df_q.merge(df_info[['number', 'start']], on='number', how='left')

df_sorted = df_q.sort_values('number')
mode_col = _detect_mode_column(df_info)

# Build mode spans
x_min = df_sorted['number'].astype(int).min()
x_max = df_sorted['number'].astype(int).max()
t_min = df_sorted['start'].min() if 'start' in df_sorted.columns else None
t_max = df_sorted['start'].max() if 'start' in df_sorted.columns else None

run_intv, time_intv, color_map, legend_h = _build_mode_spans(
    df_info, mode_col, x_min=x_min, x_max=x_max,
    time_min=t_min, time_max=t_max,
)

# Create figure
fig = plt.figure(figsize=(20, 12))
gs = gridspec.GridSpec(2, 2, width_ratios=[20, 1], hspace=0.17, wspace=0.02)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0], sharey=ax1)
cax = fig.add_subplot(gs[:, 1])

_paint_backgrounds(ax1, ax2, run_intv, time_intv, color_map)

# Split normal / bad
has_c = 'is_consensus_bad' in df_sorted.columns
df_n = df_sorted[~df_sorted['is_consensus_bad']] if has_c else df_sorted
df_b = df_sorted[df_sorted['is_consensus_bad']] if has_c else pd.DataFrame()

# Run-number axis
ax1.plot(df_sorted['number'].astype(int), df_sorted['quality_score'],
         alpha=0.4, color='gray', linewidth=1.5, zorder=1)
sc = ax1.scatter(df_n['number'].astype(int), df_n['quality_score'],
                 c=df_n['quality_score'], cmap='RdYlGn', s=45,
                 edgecolors='k', linewidth=0.5, alpha=0.9, zorder=2)
if not df_b.empty:
    ax1.scatter(df_b['number'].astype(int), df_b['quality_score'],
                c='#e74c3c', marker='X', s=120, edgecolors='black',
                linewidth=1.0, zorder=3)

ax1.set_title('SR2 LowER Detector Quality — Consensus Trend',
              pad=55, fontsize=22)
ax1.set_xlabel('Run Number', fontsize=16, labelpad=15, weight='bold')
ax1.set_ylabel('Quality Score (0–100)', fontsize=16, weight='bold')
ax1.grid(True, linestyle='--', alpha=0.6)

# Time axis
if 'start' in df_sorted.columns:
    df_t = df_sorted.sort_values('start')
    df_tn = df_t[~df_t['is_consensus_bad']] if has_c else df_t
    df_tb = df_t[df_t['is_consensus_bad']] if has_c else pd.DataFrame()
    
    ax2.plot(df_t['start'], df_t['quality_score'],
             alpha=0.4, color='gray', linewidth=1.5, zorder=1)
    ax2.scatter(df_tn['start'], df_tn['quality_score'],
                c=df_tn['quality_score'], cmap='RdYlGn', s=45,
                edgecolors='k', linewidth=0.5, alpha=0.9, zorder=2)
    if not df_tb.empty:
        ax2.scatter(df_tb['start'], df_tb['quality_score'],
                    c='#e74c3c', marker='X', s=120, edgecolors='black',
                    linewidth=1.0, zorder=3)
    ax2.set_xlabel('Time (Year-Month)', fontsize=16, labelpad=15, weight='bold')
    ax2.set_ylabel('Quality Score (0–100)', fontsize=16, weight='bold')
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.xaxis.set_major_locator(ticker.MaxNLocator(nbins=8))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax2.get_xticklabels(), rotation=0, ha='center', weight='bold', fontsize=14)

# Legend
if not df_b.empty:
    legend_h.append(plt.Line2D([0], [0], marker='X', color='w',
                                markerfacecolor='#e74c3c', markeredgecolor='black',
                                markersize=12, label='Consensus Bad'))
if legend_h:
    ax1.legend(handles=legend_h, loc='upper center', bbox_to_anchor=(0.5, 1.18),
               ncol=min(len(legend_h), 6), framealpha=1.0, edgecolor='black',
               fontsize=14, fancybox=False)

# Colorbar
cbar = fig.colorbar(sc, cax=cax)
cbar.set_label('Quality Score', weight='bold', fontsize=16)
cbar.ax.tick_params(labelsize=14)
cbar.outline.set_linewidth(3)
cbar.outline.set_edgecolor('black')
for ax in [ax1, ax2, cax]:
    for spine in ax.spines.values():
        spine.set_linewidth(3)
        spine.set_color('black')

plt.show()

### 3.3 Per-Run Diagnostic

For any run of interest, show the top-10 features that deviate most from the global mean (Z-score). Red bars indicate deviations beyond ±3σ.

In [ ]:
# Per-run Z-score diagnostic for a specific run
INSPECT_RUN = '054585'

# Find rate-based feature columns
rate_features = [c for c in df_single.columns
                 if c.endswith('_Rate_Hz') or c.endswith('_Count')]

if rate_features and INSPECT_RUN in df_single['number'].values:
    row = df_single[df_single['number'] == INSPECT_RUN]
    mean = df_single[rate_features].mean()
    std = df_single[rate_features].std().replace(0, 1e-9)
    z = (row[rate_features].iloc[0] - mean) / std
    top10 = z.abs().sort_values(ascending=False).head(10)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ['#B22222' if abs(x) > 3 else '#4682B4' for x in top10.values[::-1]]
    top10[::-1].plot(kind='barh', color=colors, edgecolor='black', linewidth=2.0, ax=ax)
    ax.axvline(0, color='black', linewidth=3)
    ax.axvline(3, color='black', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.axvline(-3, color='black', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.set_title(f'Run {INSPECT_RUN} — Top 10 Feature Deviations (Z-Score)',
                 pad=20, fontsize=16, fontweight='bold')
    ax.set_xlabel('Standard Deviations from Mean', fontsize=14, fontweight='bold')
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
        spine.set_color('black')
    plt.tight_layout()
    plt.show()
else:
    print(f'Run {INSPECT_RUN} not found or no rate features available.')

### 3.4 All Detector Features Evolution

Generate a multi-page PDF showing every numerical detector parameter over the full SR2 data-taking period. Each run is colour-coded by run number, with distinct markers per calibration type.  

**Note**: this produces a large PDF (~30+ pages). Set `RUN_FULL_EVOLUTION = True` to execute.

In [ ]:
RUN_FULL_EVOLUTION = False  # Set to True to generate the full multi-page PDF

if RUN_FULL_EVOLUTION:
    plot_all_features(
        run_info_csv=RUN_INFO_CSV,
        output_pdf='all_features_evolution_inline.pdf',
        calib_csv=os.path.join(PROJECT_DIR,
                               'split_modes/calibration_intervals_summary.csv'),
    )
    print('✓ Full-feature evolution PDF saved.')
else:
    print('Skipped. Set RUN_FULL_EVOLUTION = True to generate the full PDF.')

---
## 4. Summary

| Step | Script | Key Output |
|---|---|---|
| 1. Rate extraction | `compute_event_rates.py` | `sr2_master_run_rates.csv` |
| 2. Quality scoring | `compute_quality_scores.py` | `results/sr2_quality.h5` |
| 3. Visualization | `plot_evolution.py` | PNG/PDF plots |

### Quick reference — CLI usage

```bash
# Event rates
python compute_event_rates.py -r 054585
bash submit_batch.sh                    # batch all runs via Slurm

# Quality scores
python compute_quality_scores.py --mode consensus \
    --run-info sr2_run_tagging_info.csv \
    --rates sr2_master_run_rates.csv \
    --windows 1 2 4 8 10 --export-anomalies

# Visualization
python plot_evolution.py rates --rates sr2_master_run_rates.csv
python plot_evolution.py quality --quality results/sr2_quality.h5 --run-info ...
python plot_evolution.py features --run-info ...
python plot_evolution.py anomaly --quality ... --run-info ... --windows 1 2 4 8 10
python plot_evolution.py diagnose --run-id 054585 --quality ... --run-info ...
```

### Data types and colour scheme

| Type | Colour | Description |
|---|---|---|
| Science Run | `#F3F3EF` | Background / physics data |
| Kr-83m | `#FF0026` | Krypton-83m calibration |
| Rn-220 | `#FF00FF` | Radon-220 calibration |
| AmBe | `#32CD32` | Americium-Beryllium neutron source |
| Ar-37 | `#FF8C00` | Argon-37 calibration |
| Neutron | `#8B4513` | Neutron generator |
| Th-232 | `#100CCE` | Thorium-232 calibration |